# Elasticsearch and OpenSearch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/elasticsearch.ipynb)

*Generated from `content/elasticsearch.md` for Google Colab.* Markdown explanations are preserved, and fenced examples are converted into executable notebook cells wherever a Python kernel can run them. Some cells still require the right data, tables, credentials, Java runtime, Spark runtime, or cloud account before they can succeed.

## How to run

- Open this notebook in Google Colab using the badge above.
- Run the setup cell first.
- Run cells one by one and replace sample table names, file paths, credentials, and cloud resources with your own.
- SQL cells use `%%sql` with an in-memory SQLite connection unless you configure another database.
- PySpark cells need `pyspark` and Java. Colab usually has Java; install PySpark from the setup hint if missing.

Notebook inventory: **111 markdown cells**, **13 code cells**. Detected dependencies: `pyarrow`.

In [ ]:
# Colab setup: run this cell first.
import importlib.util
import sys

required_packages = ['pyarrow']
package_modules = {'apache-airflow': 'airflow', 'delta-spark': 'delta', 'google-cloud-storage': 'google.cloud.storage', 'ipython-sql': 'sql'}
print("Python:", sys.version.split()[0])

for package in required_packages:
    module = package_modules.get(package, package.split("-")[0])
    status = "available" if importlib.util.find_spec(module) else "missing"
    print(f"{package:22} {status}")

print("\nIf a required package is missing, run:")
print('!pip -q install pyarrow')

if "ipython-sql" in required_packages:
    try:
        get_ipython().run_line_magic("load_ext", "sql")
    except Exception:
        print("Install ipython-sql, restart the runtime if needed, then rerun this setup cell.")
    else:
        get_ipython().run_line_magic("sql", "sqlite://")
        print("SQL magic connected to an in-memory SQLite database.")

Elasticsearch is a distributed search and analytics engine built on Apache Lucene. Documents are JSON, they live in indices split into shards and replicas, and nearly every query is answered from an inverted index rather than a table scan. OpenSearch is the Apache-2.0 fork of Elasticsearch 7.10.2, so the mental model and most of the API carry over; the differences are licensing, plugins, and feature velocity.

For a data engineer the framing is: Elasticsearch is a serving layer for text search, log analytics, faceted aggregation, and low-latency filtered lookups. It is not a system of record and not a warehouse, and treating it as either is the fastest route to an expensive, fragile cluster.

## What is Elasticsearch?

Elasticsearch stores JSON documents and answers two questions unusually fast: which documents contain these terms, and what are the counts and metrics grouped by these fields. It is built on Lucene segments, scales horizontally by splitting an index into shards, and tolerates node loss by keeping replicas on other nodes.

In [ ]:
%%bash
curl -s localhost:9200/_cluster/health?pretty
curl -s localhost:9200/_cat/indices?v

Name these in an interview: a cluster of nodes, indices made of primary and replica shards, a mapping of field types, analyzers that tokenize text, the inverted index for search, doc values for sorting and aggregation, and `_bulk` plus aliases for ingestion and zero-downtime changes. Since 7.11 the default distribution is Elastic License 2.0 / SSPL; OpenSearch forked the last Apache-2.0 release, 7.10.2.

## How are indices shards and replicas organized?

An index is a logical namespace split into primary shards, each an independent Lucene index on one node. Replicas copy a primary onto another node, serve reads, and take over on failure. Primary shard count is fixed at creation; replicas can change at any time.

In [ ]:
%%bash
curl -X PUT localhost:9200/orders-v1 -H 'Content-Type: application/json' -d '{
  "settings": {"number_of_shards": 6, "number_of_replicas": 1, "refresh_interval": "5s"}
}'

curl -X PUT localhost:9200/orders-v1/_settings -H 'Content-Type: application/json' \
  -d '{"index": {"number_of_replicas": 2}}'
curl -s 'localhost:9200/_cat/shards/orders-v1?v'

Sizing rules: one shard per 10-50 GB of data, shard count a small multiple of the number of data nodes, and total shards per node in the low hundreds because each shard carries heap and file-handle overhead. Replicas cost disk and write bandwidth but buy availability and read throughput, and a primary and its replica never share a node.

## How do mappings work and what is the difference between keyword and text?

The mapping is the schema: it declares each field's type and how it is indexed. Indexing an identifier as `text` and then filtering or aggregating on it is the classic mistake. `text` is analyzed into tokens and searched with relevance; `keyword` is stored verbatim for exact filtering, sorting, and aggregation.

```json
{
  "mappings": {
    "properties": {
      "order_id": { "type": "keyword" },
      "status": { "type": "keyword" },
      "description": { "type": "text", "analyzer": "english" },
      "total": { "type": "scaled_float", "scaling_factor": 100 },
      "created_at": { "type": "date" },
      "shipping": { "properties": {
        "city": { "type": "keyword" }, "location": { "type": "geo_point" } } }
    }
  }
}
```

When a field must serve both purposes, use a multi-field: `"description": {"type": "text", "fields": {"raw": {"type": "keyword"}}}` indexes the value twice. Other types worth naming: `long`, `double`, `scaled_float` for money, `date`, `boolean`, `ip`, `geo_point`, `nested` for arrays of objects that must not cross-match, and `flattened` for arbitrary key/value blobs.

## What are analyzers and how do they affect search?

An analyzer runs at index time and again at query time: character filters, then a tokenizer, then token filters. The analyzers to know are `standard` (Unicode word boundaries, lowercasing), `keyword` (the whole value as one token), and `whitespace` for special cases. A mismatch between index and query analysis is a top cause of "the document exists but search returns nothing".

In [ ]:
%%bash
curl -s 'localhost:9200/_analyze?pretty' -H 'Content-Type: application/json' -d '{
  "analyzer": "standard", "text": "The 3 Quick Brown Foxes"
}'

curl -X PUT localhost:9200/docs -H 'Content-Type: application/json' -d '{
  "settings": {"analysis": {"analyzer": {"html_english": {
    "type": "custom", "tokenizer": "standard", "char_filter": ["html_strip"],
    "filter": ["lowercase", "stop", "porter_stem"] }}}},
  "mappings": {"properties": {"body": {"type": "text", "analyzer": "html_english"}}}
}'

The subtlety: `match` runs the analyzer on user input, but `term` does not, so `term` on a `text` field almost always fails to match. Use `match` for analyzed text and `term` only on `keyword`, numeric, or date values.

## What are the pitfalls of dynamic mapping?

With dynamic mapping on, the first document defines the types, and a mixed field can silently produce a mapping you cannot change. A `long` cannot later store a decimal, and a field first seen as a number cannot accept text without reindexing.

In [ ]:
%%bash
# This fails once total is mapped as a number
curl -X POST localhost:9200/orders/_doc -H 'Content-Type: application/json' \
  -d '{"total": "unknown"}'
# mapper_parsing_exception: failed to parse field [total] of type [float]

Mitigate by disabling or limiting dynamic mapping in the template (`"dynamic": "strict"` for known schemas, `"dynamic": false` to ignore unknown fields), pre-creating templates for every dataset, and routing unstructured payloads into a `flattened` field. Also remember that every string defaults to `text` plus a keyword sub-field, which doubles index storage; map ids and enums explicitly as `keyword`.

## How does the inverted index work?

For each analyzed field, Lucene writes a term dictionary mapping every token to the documents containing it, with frequencies and positions. A query looks up terms and intersects or unions posting lists, which is why full-text search is fast regardless of corpus size. Separately, `doc_values` store values column-wise for sorting and aggregation, and `_source` stores the original JSON.

```text
Document 1: "red running shoes"    Document 2: "blue running jacket"    Document 3: "red jacket"

Consequences: the inverted index only helps if the query uses indexed terms, which is why wildcards and leading-`*` regexes are slow (they scan terms) and why `text` fields cannot be sorted. Segment merges rewrite these structures, so indexing is I/O heavy and deletes are tombstones until a merge.

## How do you index and update documents with the CRUD and bulk APIs?

Single-document calls are fine interactively; every pipeline should use `_bulk`, where each line is an action (`index`, `create`, `update`, `delete`) followed by the source, newline-delimited and without commas.

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

es = Elasticsearch("http://localhost:9200")


def actions(rows):
    for row in rows:
        yield {
            "_op_type": "update",
            "_index": "orders",
            "_id": row["order_id"],
            "doc": {"status": row["status"], "updated_at": row["updated_at"]},
            "doc_as_upsert": True,
        }


def load(rows) -> tuple[int, list]:
    return bulk(es, actions(rows), chunk_size=2000, request_timeout=120)

Indexing is near-real-time because a refresh makes new segments searchable, so a 30s `refresh_interval` and `refresh=false` during a backfill are much faster. A `200` from `_bulk` does not mean every item succeeded: inspect `errors` and retry `429`/`503` with backoff. Use `create` for exactly-once insertion and `update` with `doc_as_upsert` for idempotent upserts.

## How do you write search queries with match and term?

`match` analyzes the input and is the default for text; `term` matches one exact token and is for `keyword`, numbers, and booleans; `multi_match` searches several fields and combines their scores.

```json
{
  "query": {
    "bool": {
      "must": [
        { "multi_match": {
            "query": "wireless noise cancelling",
            "fields": ["title^3", "description", "brand^2"],
            "type": "best_fields"
        } }
      ],
      "filter": [
        { "term": { "status": "in_stock" } },
        { "range": { "price": { "gte": 50, "lte": 500 } } }
      ]
    }
  }
}
```

The `^3` boost makes title matches matter more, a first-order relevance lever before any ranking pipeline. `best_fields` scores the single best field, `most_fields` sums across fields, and `cross_fields` treats several fields as one, which is right for a first and last name searched together. Range queries support date math such as `"gte": "now-7d/d"`.

## How do bool queries combine clauses?

A `bool` query has four clause types: `must` (scores and filters), `filter` (filters, no score, cacheable), `should` (optional, contributes score, and with `minimum_should_match` becomes required), and `must_not` (excludes).

```json
{
  "query": {
    "bool": {
      "must": [ { "match": { "description": "espresso machine" } } ],
      "filter": [
        { "term": { "category": "kitchen" } },
        { "terms": { "warehouse": ["DFW", "PHX"] } },
        { "range": { "stock": { "gt": 0 } } }
      ],
      "should": [
        { "term": { "tags": "coffee" } },
        { "term": { "tags": "sale" } }
      ],
      "minimum_should_match": 1,
      "must_not": [ { "term": { "status": "discontinued" } } ]
    }
  }
}
```

The usual shape is `must` for relevance text, `filter` for yes/no attributes, and `should` for boosts. Because `filter` clauses skip scoring and are cacheable, moving numeric and term constraints out of `must` makes a query both faster and cache-friendly.

## What is the difference between queries and filters?

A query clause computes a BM25 `_score`, which is more expensive and depends on term statistics. A filter clause is a yes/no predicate: no score, and results cacheable in the filter cache. Elastic's own guidance is to filter anything that does not need to affect relevance.

```json
{
  "query": {
    "bool": {
      "must": [ { "match": { "title": "data engineering" } } ],
      "filter": [
        { "term": { "tenant_id": "t-88" } },
        { "term": { "doc_type": "article" } },
        { "range": { "published_at": { "gte": "2026-01-01" } } }
      ]
    }
  },
  "track_total_hits": false
}
```

Small choices compound: `track_total_hits: false` avoids counting every hit, `_source: false` or a `fields` list avoids shipping whole documents, and a selective `filter` lets the engine skip posting lists entirely.

## How do aggregations work in Elasticsearch?

Aggregations compute metrics (`avg`, `sum`, `min`, `max`, `percentiles`, `cardinality`) and buckets (`terms`, `date_histogram`, `range`, `histogram`). Buckets nest, and a terms bucket can carry sub-aggregations to build a report in one request.

```json
{
  "size": 0,
  "query": { "range": { "created_at": { "gte": "now-7d/d" } } },
  "aggs": {
    "by_day": {
      "date_histogram": { "field": "created_at", "calendar_interval": "day",
        "time_zone": "America/Chicago" },
      "aggs": { "revenue": { "sum": { "field": "total" } },
        "p95": { "percentiles": { "field": "total", "percents": [50, 95, 99] } } }
    },
    "top_customers": {
      "terms": { "field": "customer_id", "size": 10, "order": { "revenue": "desc" } },
      "aggs": { "revenue": { "sum": { "field": "total" } } }
    },
    "unique_customers": { "cardinality": { "field": "customer_id" } }
  }
}
```

Caveats an interviewer probes: `terms` on a high-cardinality field is approximate - the top N per shard are merged, so `doc_count_error_upper_bound` can be non-zero; raise `shard_size` or use a composite aggregation to page all buckets. `cardinality` is approximate HyperLogLog++. `date_histogram` needs `time_zone` for local buckets, and `size: 0` is mandatory when only aggregations are wanted.

## How does pagination work and what are the from and size limits?

Deep pagination with `from`/`size` is capped by `index.max_result_window` (default 10,000) because every shard fetches and sorts `from + size` hits. Beyond that, use `search_after` with a stable sort, pinned to a point-in-time (PIT) so the sort order cannot shift as documents are indexed.

In [ ]:
%%bash
curl -X POST 'localhost:9200/orders/_pit?keep_alive=2m'
# {"id":"48my..."}

curl -X POST localhost:9200/_search -H 'Content-Type: application/json' -d '{
  "size": 100,
  "pit": { "id": "48my...", "keep_alive": "2m" },
  "sort": [ { "created_at": "desc" }, { "_id": "asc" } ],
  "search_after": [ "2026-10-04T13:00:00.000Z", "1000" ]
}'

curl -X DELETE 'localhost:9200/_pit' -H 'Content-Type: application/json' -d '{"id":"48my..."}'

Use `scroll` only for bulk export; it holds segments open and is deprecated in favor of PIT plus `search_after`. For exports, writing a PIT result to Parquet is usually cheaper than paging it to a user, and `from`/`size` should stay shallow for interactive UIs.

## What are index templates and how do you use them?

A template applies settings, mappings, and aliases to every index whose name matches a pattern. Composable and component templates let shared settings be defined once and overridden per dataset, which is how a logging platform creates correctly mapped daily indices without humans in the loop.

In [ ]:
%%bash
curl -X PUT localhost:9200/_index_template/logs-app -H 'Content-Type: application/json' -d '{
  "index_patterns": ["logs-app-*"],
  "priority": 200,
  "template": {
    "settings": {
      "number_of_shards": 3, "number_of_replicas": 1, "refresh_interval": "10s",
      "index.lifecycle.name": "logs-ilm",
      "index.default_pipeline": "logs-parse"
    },
    "mappings": {
      "dynamic": "strict",
      "properties": {
        "@timestamp": { "type": "date" }, "service": { "type": "keyword" },
        "level": { "type": "keyword" }, "message": { "type": "text" }
      }
    }
  }
}'

Templates are resolved by priority and pattern specificity, so overlapping templates are a common source of "why did my mapping come out wrong". `_simulate_index` shows the resolved settings before the first document arrives, and an ingest pipeline attached in the template parses every document automatically.

## How do aliases and reindex work?

An alias is a second name for one or more indices. Writing through an alias with `is_write_index` and searching through an alias that fans out over several indices is the standard pattern: applications never hard-code an index name, so a rebuild is invisible.

In [ ]:
%%bash
curl -X POST localhost:9200/_reindex?wait_for_completion=false -H 'Content-Type: application/json' -d '{
  "source": { "index": "orders-v1" },
  "dest": { "index": "orders-v2", "op_type": "create" }
}'

curl -X POST localhost:9200/_aliases -H 'Content-Type: application/json' -d '{
  "actions": [
    { "remove": { "index": "orders-v1", "alias": "orders" } },
    { "add": { "index": "orders-v2", "alias": "orders", "is_write_index": true } }
  ]
}'

`_reindex` copies documents and applies the destination mapping, so it is also the migration path after a mapping change - a field's type cannot be changed in place. It is not incremental by itself; for live data, pause writes or dual-write while the reindex catches up, then flip. `slices` parallelizes a large reindex.

## How does index lifecycle management work?

ILM moves an index through hot, warm, cold, frozen, and delete phases based on age or size. Rollover creates a new backing index at a threshold, and each phase can change replicas, force-merge, move to cheaper nodes, or shrink.

```json
{
  "policy": {
    "phases": {
      "hot": { "actions": {
        "rollover": { "max_primary_shard_size": "50gb", "max_age": "1d" },
        "set_priority": { "priority": 100 } } },
      "warm": { "min_age": "2d", "actions": {
        "forcemerge": { "max_num_segments": 1 }, "set_priority": { "priority": 50 } } },
      "cold": { "min_age": "30d", "actions": {
        "readonly": {}, "searchable_snapshot": {}
      } },
      "delete": { "min_age": "365d", "actions": { "delete": {} } }
    }
  }
}
```

Attach it with `"index.lifecycle.name": "logs-ilm"` in the template and write through a rollover alias. Hot nodes need fast NVMe and CPU, warm nodes cheaper disk, and cold/frozen tiers keep data searchable from object storage. `searchable_snapshot` is the biggest cost lever for log retention because it removes the local copy.

## How do you ingest data with Logstash Beats and Kafka Connect?

There are four common paths: Beats agents ship logs and metrics; Logstash parses, enriches, and fans out; Kafka Connect sinks records from Kafka; and small services call the bulk API directly. For a data platform, Kafka plus a sink is usually most robust because Kafka absorbs spikes and replay handles index rebuilds.

```yaml
input:
  kafka:
    bootstrap_servers: "broker1:9092,broker2:9092"
    topics: ["logs.app"]
    codec: json
filter:
  date:
    match: ["ts", "ISO8601"]
    target: "@timestamp"
output:
  elasticsearch:
    hosts: ["https://es.internal:9200"]
    index: "logs-app-%{+YYYY.MM.dd}"
    ilm_enabled: true
    manage_template: true
```

```text
Beats --> Logstash --> Elasticsearch
Kafka --> Kafka Connect (Elasticsearch sink, bulk, retries) --> Elasticsearch
CDC (Debezium) --> Kafka --> sink --> Elasticsearch
```

Kafka Connect's sink batches, retries, and supports exactly-once with a dead-letter queue for rejected records. Use a business key as the document `_id` (for example `order_id`) so replays overwrite rather than duplicate, and always validate an ingest pipeline with `_ingest/pipeline/_simulate` before pointing production traffic at it.

## How does OpenSearch differ from Elasticsearch?

OpenSearch is the Apache-2.0 fork of Elasticsearch 7.10.2 plus OpenSearch Dashboards. Core concepts, mapping types, the DSL, and the bulk API are compatible enough that knowledge transfers; the differences are at the edges.

| Aspect | Elasticsearch | OpenSearch |
|---|---|---|
| License | Elastic License 2.0 / SSPL | Apache License 2.0 |
| Origin | Elastic / Lucene | AWS-led fork of 7.10.2 |
| Vector search | dense_vector, kNN, ELSER | kNN, neural search plugin |
| SQL | SQL, ESQL | SQL and PPL |
| Alerting | Kibana alerting | Alerting plugin, anomaly detection |
| Dashboards | Kibana | OpenSearch Dashboards |
| Managed | Elastic Cloud, AWS (limited) | AWS OpenSearch Service, self-hosted |

```text
GET _cluster/health                 # same in both
POST _plugins/_knn/{index}/_search  # OpenSearch plugin endpoint
```

OpenSearch wins when the license, an AWS-native managed service, or PPL/SQL querying matters; Elasticsearch wins on the latest relevance and vector features. Both keep the same operational shape - shards, mappings, templates, ILM (ISM in OpenSearch), and snapshots - so a migration is mostly reindexing and re-creating dashboards and alerts, not re-learning search.

## How do you size an Elasticsearch cluster and plan shards?

Sizing starts from data volume, retention, and query shape. Estimate raw JSON size, multiply for `_source` plus indexes, add replicas and headroom, and divide into shards of 10-50 GB. Memory is dominated by the JVM heap (cap around 31 GB to keep compressed ordinary object pointers) and by the filesystem cache, which should hold the hot working set.

```text
1 TB/day raw JSON, 30-day retention
  stored/day          ~ 1 TB * 0.40 compression = 400 GB primary
  with one replica    ~ 800 GB/day
  30 days             ~ 24 TB per replica set (before cold/frozen offload)
  shards at 40 GB     ~ 10 primary shards/day, rolled over daily

Rules that survive production: make shards a few tens of GB, avoid tens of thousands of tiny daily shards, roll over at `max_primary_shard_size` rather than a fixed daily count when volume varies, and keep at least one replica. Over-sharding is far more common than under-sharding and costs heap, merge threads, and cluster state.

## How do you profile and tune query performance?

Start with the slow log and the profile API, not guesswork. Search is slow for recurring reasons: a wildcard or script query, deep `from`/`size`, a high-cardinality `terms` aggregation, a field without `doc_values`, or a query that fans out to every shard.

In [ ]:
%%bash
curl -X PUT localhost:9200/orders/_settings -H 'Content-Type: application/json' \
  -d '{"index.search.slowlog.threshold.query.warn": "2s"}'

curl -X POST 'localhost:9200/orders/_search?pretty' -H 'Content-Type: application/json' \
  -d '{"profile": true, "query": {"bool": {"must": [{"match": {"description": "espresso"}}]}}}'

curl -s 'localhost:9200/_nodes/stats/indices/search,indices/query_cache?pretty'

Fixes by payoff: move non-scoring constraints into `filter`, replace wildcards with an analyzed field or `keyword` prefix, add `track_total_hits: false`, reduce `size` and use `search_after`, pre-aggregate with transforms or rollups, and raise `refresh_interval` during heavy ingestion. Force-merging read-only indices reduces segment count and improves latency, but is expensive and only safe on indices no longer written.

## How do snapshots and backups work?

A snapshot is a point-in-time copy of indices and cluster state into a repository such as S3, GCS, Azure Blob, or a shared filesystem. Snapshots are incremental, so unchanged segments are referenced rather than recopied, which makes frequent snapshots cheap. It is the only sanctioned backup mechanism; copying data directories is not.

In [ ]:
%%bash
curl -X PUT localhost:9200/_snapshot/s3-backups -H 'Content-Type: application/json' \
  -d '{"type": "s3", "settings": {"bucket": "acme-es-backups", "base_path": "prod"}}'

curl -X PUT 'localhost:9200/_snapshot/s3-backups/snap-2026-10-04?wait_for_completion=true' \
  -H 'Content-Type: application/json' -d '{
    "indices": "orders,logs-app-*", "include_global_state": true, "ignore_unavailable": true
  }'

curl -X POST localhost:9200/_snapshot/s3-backups/snap-2026-10-04/_restore \
  -H 'Content-Type: application/json' \
  -d '{"indices": "orders", "rename_pattern": "orders", "rename_replacement": "orders_restored"}'

Add a `snapshot` action to ILM for long retention, and run a snapshot lifecycle policy (SLM) on a schedule with a retention count. Always test a restore into a differently named index, and keep the repository in a separate account or region so one credential mistake cannot remove both copies.

## How do you secure an Elasticsearch cluster?

Security has four layers: transport and HTTP TLS, authentication, authorization (roles plus document and field-level access), and audit logging. Exposing an unauthenticated cluster to a network is a breach waiting for a scanner.

```yaml
xpack.security.enabled: true
xpack.security.transport.ssl.enabled: true
xpack.security.transport.ssl.verification_mode: certificate
xpack.security.transport.ssl.keystore.path: certs/elastic-certificates.p12
xpack.security.http.ssl.enabled: true
xpack.security.http.ssl.keystore.path: certs/http.p12
```

In [ ]:
%%bash
curl -X POST localhost:9200/_security/role/logs_reader -H 'Content-Type: application/json' \
  -d '{"indices": [{"names": ["logs-app-*"], "privileges": ["read"]}]}'

curl -X POST localhost:9200/_security/role/tenant_t88 -H 'Content-Type: application/json' \
  -d '{"indices": [{"names": ["tickets-*"], "privileges": ["read"],
        "query": {"term": {"tenant_id": "t-88"}}}]}'

Beyond TLS and roles: API keys rather than shared passwords for services, field-level security and masked fields for PII, the audit log shipped off-cluster, encrypted snapshot repositories, and the cluster bound to a private network. When one index holds several tenants, document-level security plus a strict template is the isolation mechanism until tenants move to separate indices.

## How do you design a log analytics architecture?

A log platform is a pipeline with a hot serving tier and a cold archive tier: producers write JSON with a timestamp and service identifier, shippers buffer and compress, a processing tier parses and enriches, and the index tier serves a bounded window while older data lives in object storage.

```text
apps --> Fluent Bit/Filebeat --> Kafka --> Logstash/Connect --> Elasticsearch
                                      |                              |
                                      |                              +-- ILM hot -> warm -> frozen -> delete
                                      +--> S3 archive (Parquet) --> Trino/Athena

Decisions that matter: normalize timestamp and service in the template so every dashboard works across indices; keep identifiers as `keyword` and free text as `text`; use `dynamic: strict` so a new field cannot blow up the mapping; set `refresh_interval` to 10-30s because dashboards do not need sub-second freshness; and sample or drop debug logs at the shipper. The Parquet archive is not optional - search clusters are expensive per byte, and Parquet answers compliance retention at a fraction of the cost.

## How does search relevance work?

Relevance is the BM25 `_score`, which rewards term frequency, penalizes common terms, and normalizes by document length. The practical levers in order: get the analyzer right, use `multi_match` with field boosts, add `should` clauses for recency or popularity, use `function_score` for business signals, and only then consider learning-to-rank.

```json
{
  "query": {
    "function_score": {
      "query": { "multi_match": { "query": "espresso machine", "fields": ["title^3", "description"] } },
      "functions": [
        { "filter": { "term": { "in_stock": true } }, "weight": 2 },
        { "gauss": { "published_at": { "origin": "now", "scale": "30d", "decay": 0.5 } } }
      ],
      "score_mode": "sum",
      "boost_mode": "multiply"
    }
  }
}
```

Text relevance and analytics pull in opposite directions: the fields you analyze for search must also exist as `keyword` for aggregation. Measure relevance with a judgment set and nDCG or MRR rather than eyeballing, and use `explain: true` when a score is surprising. For zero-result queries, fall back to prefix or fuzzy matching (`fuzziness: "AUTO"`) and log the query so the gap is visible.

## Elasticsearch vs ClickHouse vs Parquet and Trino?

All three answer analytical questions at different points on the latency/retention/operability curve. The choice follows the query shape: text search and faceted navigation go to Elasticsearch, high-concurrency columnar aggregation to ClickHouse, and cheap large-scale retention to Parquet queried by Trino.

| Dimension | Elasticsearch / OpenSearch | ClickHouse | Parquet + Trino |
|---|---|---|---|
| Storage | inverted index + doc values | columnar MergeTree parts | columnar immutable files |
| Best at | text search, facets, logs | sub-second aggregations | large scans, open data |
| Full-text | excellent, analyzers, relevance | basic token indexes | none natively |
| Updates | per document, near-real-time | async mutations | rewrite or delete files |
| Concurrency | high for search, agg-heavy costly | very high for aggregations | high via a cluster |
| Storage cost | highest | medium | lowest |
| Retention | ILM tiers, expensive per byte | local disks or S3 tiered | effectively unlimited |
| SQL | SQL/ESQL, limited joins | strong SQL, limited joins | full SQL and joins |
| Ops | JVM cluster, shard tuning | self-managed tuning | stateless query layer + metastore |

```text
search + logs -> Elasticsearch    aggregations -> ClickHouse    history -> Parquet + Trino
```

A defensible architecture uses all three: Kafka to Parquet for cheap immutable history, Trino or Spark for batch analytics over it, ClickHouse or a warehouse for interactive dashboards, and Elasticsearch for search, log exploration, and faceted filtering. Choosing one for everything is how teams end up with a search cluster doing warehousing.

## Real world use cases for Elasticsearch

### 1. Log analytics and observability

Business context: 4,000 services emit 1.5 TB of logs/day; engineers search the last 7 days in under a second and compliance needs 12 months. Design: Filebeat to Kafka, Connect for parsing into a strict template, daily rollover indices with ILM, and a Parquet archive in S3.

```json
{
  "index_patterns": ["logs-app-*"],
  "template": {
    "settings": { "refresh_interval": "10s", "index.lifecycle.name": "logs-ilm" },
    "mappings": {
      "dynamic": "strict",
      "properties": {
        "@timestamp": { "type": "date" }, "service": { "type": "keyword" },
        "level": { "type": "keyword" }, "message": { "type": "text" }
      }
    }
  }
}
```

Why Elasticsearch: analyzed text search plus `terms` and `date_histogram` answer both "find this error" and "how many per service per hour", and ILM moves older data to cheaper tiers.

### 2. E-commerce search with faceted navigation

Business context: 8M SKUs, 12,000 searches/s, filters on brand, price, and rating, with continuous catalog updates. Design: `text` for title and description, `keyword` for brand, `scaled_float` for price, Kafka Connect with the SKU as `_id`, and an alias for reindex.

```json
{
  "size": 20,
  "query": {
    "bool": {
      "must": [ { "multi_match": { "query": "noise cancelling headphones",
        "fields": ["title^3", "brand^2", "description"] } } ],
      "filter": [ { "term": { "in_stock": true } },
        { "range": { "price": { "gte": 50, "lte": 400 } } } ]
    }
  },
  "aggs": {
    "brands": { "terms": { "field": "brand", "size": 20 } },
    "avg_rating": { "avg": { "field": "rating" } }
  }
}
```

Why Elasticsearch: one request returns ranked hits and the filter counts the UI needs, and boosting plus `function_score` lets merchandising influence order.

### 3. Security information and event management

Business context: a security team correlates events from 60 sources and needs continuous detection plus forensic search over 13 months. Design: normalized events, a write alias with rollover, ILM with a frozen tier, scheduled detection searches, and immutable snapshots.

```text
detection rule, every 5 minutes
  search:   event.category:authentication AND event.outcome:failure
  group by: source.ip
  trigger:  count > 20 within 5 minutes
```

Why Elasticsearch: high-cardinality keyword filtering plus aggregations make correlation practical, and each hit keeps its `_source` for investigation.

### 4. Product catalog sync from a lakehouse

Business context: the warehouse (Iceberg on S3, queried by Trino) is the system of record; the search index must reflect changes within minutes and survive a rebuild without downtime. Design: a curated Parquet export, a bulk-index job into a versioned index, and an alias flip.

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import streaming_bulk
import pyarrow.dataset as ds

es = Elasticsearch("https://es.internal:9200")


def documents(path: str, batch_size: int = 5000):
    dataset = ds.dataset(path, format="parquet")
    for batch in dataset.to_batches(batch_size=batch_size):
        for row in batch.to_pylist():
            yield {"_index": "products-v3", "_id": row["sku"], "_source": row}


def rebuild(path: str) -> tuple[int, list]:
    return streaming_bulk(es, documents(path), chunk_size=2000, max_retries=5)

Why Elasticsearch: bulk indexing with business-key ids is idempotent, the alias makes the swap invisible, and the lakehouse stays the single source of truth.

### 5. Multi-tenant customer support search

Business context: a SaaS helpdesk holds 900M tickets across 2,000 tenants, every query is tenant-scoped, and one tenant is 30% of the data. Design: a shared index with `tenant_id` as `keyword`, document-level security, routing by `tenant_id`, and per-contract ILM retention.

```json
{
  "query": {
    "bool": {
      "must": [ { "multi_match": { "query": "refund not received",
        "fields": ["subject^2", "body"] } } ],
      "filter": [
        { "term": { "tenant_id": "t-88" } },
        { "range": { "created_at": { "gte": "now-90d/d" } } }
      ]
    }
  },
  "highlight": { "fields": { "body": {} } }
}
```

In [ ]:
%%bash
curl -X POST 'localhost:9200/tickets/_doc/9001?routing=t-88' -H 'Content-Type: application/json' \
  -d '{"tenant_id":"t-88","subject":"Refund missing","body":"I have not received my refund."}'

Why Elasticsearch: document-level security plus routing gives tenant isolation and locality, highlight returns the snippet the UI needs, and per-tenant retention is an ILM concern.